[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/thermodynamics/property_charts/property_charts.ipynb)

# Air Conditioning on the Psychrometric Chart

This example computes the cooling, the water removed and the reheating needed to bring warm humid air to a comfortable supply condition. A cooling coil cools the air until it is saturated and then cools it further while the water beyond saturation condenses on the coil; the air leaves cold and saturated, so a reheater warms it at fixed water content to the supply temperature. The psychrometric chart, temperature against water content with lines of equal relative humidity and enthalpy, is the engineer's map of the process.

A state of the air is modelled as an antivector of the plane spanned by its temperature, `mv.t.dual()`, and its water content per kilogram of dry air, `mv.w.dual()`, each read back by pairing. Relative humidity and enthalpy are two more readings of the state, and the chart's curves are their level lines. Each stage of the process holds one reading of the change of state: the water content while the air is dry, the relative humidity at one while it condenses. Stacked with the temperature's change, the held reading is one map from a change of state to both their changes, and solving it for one degree warmer with the held reading unchanged gives the direction the air moves in; it is the same construction for either stage.

In the usual notation the enthalpy per kilogram of dry air is $h = c_{p,a} T + w (h_{fg} + c_{p,v} T)$, the coil's duty is $q = h_1 - h_2 - (w_1 - w_2)\,c_w T_2$, and the reheat $h_3 - h_2$; saturation follows the Magnus approximation, $p_\text{sat} = 0.61078\, e^{17.27 T/(T + 237.3)}$ kPa.

Temperature is in degrees Celsius, pressure in kPa, water content in kg of water per kg of dry air (on the chart, grams per kg), and enthalpy in kJ per kg of dry air.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples,jax] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import jax
import numpy as np

from numga import Algebra, concatenate, stack
from numga.backend.jax import JaxContext, derivative
from examples.util.animation import show_animation
from examples.thermodynamics.property_charts import render

jax.config.update('jax_enable_x64', True)
ga = Algebra('t0w0')
mv = JaxContext(ga, np.float64).multivector
Scalar = ga.gatype.scalar()
State = ga.gatype.antivector()
Reading = ga.gatype((Scalar, State))

# The two readings a state is described by.
temperature: Reading = mv.t & State                       # Scalar <- State
humidity: Reading = mv.w & State                          # Scalar <- State
# The pressure; the Magnus approximation of the saturation pressure; the ratio of the molar masses of water and dry
# air; the specific heats of dry air, of vapour and of liquid water, and the latent heat of vaporisation at zero degrees.
pressure = 101.325
magnus_pressure, magnus_slope, magnus_offset = 0.61078, 17.27, 237.3
molar_ratio = 0.62198
dry_air_heat, vapour_heat, liquid_heat, latent_heat = 1.006, 1.86, 4.186, 2501.0

## The chart

The saturation pressure grows exponentially with the temperature; the relative humidity is the vapour's partial pressure over it, and the enthalpy the dry air's heat plus the vapour's, latent and sensible. The chart shows their level lines over the plane of states: the lines of equal relative humidity, faint, with the saturation curve bold, and the dashed lines of equal enthalpy, marked with their values where they meet the saturation curve. Above the saturation curve, shaded, no air holds that much water as vapour. The arrows are the directions computed in the next section.

In [ ]:
def saturation_pressure(state: State) -> Scalar:
    return magnus_pressure * (magnus_slope * temperature(state) / (temperature(state) + magnus_offset)).exp()

def relative_humidity(state: State) -> Scalar:
    """The vapour's partial pressure over the saturation pressure."""
    water = humidity(state)
    return pressure * water / (molar_ratio + water) / saturation_pressure(state)

def enthalpy(state: State) -> Scalar:
    """The dry air's heat and the vapour's, latent and sensible, per kg dry air."""
    warmth, water = temperature(state), humidity(state)
    return dry_air_heat * warmth + water * (latent_heat + vapour_heat * warmth)

def air_state(warmth: float, relative: float) -> State:
    """Air at a temperature and relative humidity."""
    dry = mv.t.dual() * warmth + mv.w.dual() * 0
    vapour = relative * saturation_pressure(dry)
    return dry + mv.w.dual() * (molar_ratio * vapour / (pressure - vapour))

def constrained_direction(state: State, constraint: Reading) -> State:
    """The change of state per degree warmer with the constraint's reading held."""
    readings = stack((temperature.broadcast_to(state.shape), constraint), axis=-1).field(0)
    requested = mv.scalar([[1], [0]]).field(0)
    return readings.solve(requested)

# The inlet: warm humid air; the coil's outlet: saturated at the coil's temperature.
inlet = air_state(30.0, 0.7)                                                         # [] State
coil_temperature, supply_temperature = 12.0, 22.0
dry = mv.t.dual() * coil_temperature
outlet = dry + mv.w.dual() * (molar_ratio * saturation_pressure(dry) / (pressure - saturation_pressure(dry)))   # [] State
# Holding the water content at the inlet, holding the relative humidity at the outlet.
directions = stack((constrained_direction(inlet, humidity),
                    constrained_direction(outlet, derivative(relative_humidity)(outlet))))   # [2] State
render.draw_chart(relative_humidity, enthalpy, stack((inlet, outlet)), directions)

## Through the coil and the reheater

The coil cools the air toward its outlet temperature; at each temperature the water beyond what saturated air holds has condensed, `humidity(inlet) - saturation_humidity`, which is nothing until the air reaches its dew point. The reheater warms the air at the outlet's water content. The coil's duty is the enthalpy the air loses less the enthalpy the condensed water carries off at the outlet's temperature.

In [ ]:
def saturation_humidity(state: State) -> Scalar:
    vapour = saturation_pressure(state)
    return molar_ratio * vapour / (pressure - vapour)

n_steps = 32
fraction = np.linspace(0, 1, n_steps)
cooled_dry = mv.t.dual() * (30.0 * (1 - fraction) + coil_temperature * fraction) + mv.w.dual() * humidity(inlet)
cooled = cooled_dry - mv.w.dual() * (humidity(inlet) - saturation_humidity(cooled_dry)).clip(0, np.inf)   # [steps] State
reheated = mv.t.dual() * (coil_temperature * (1 - fraction) + supply_temperature * fraction) + mv.w.dual() * humidity(outlet)
path = concatenate([cooled, reheated[1:]])                                           # [2 * steps - 1] State

condensate = humidity(inlet) - humidity(outlet)                                      # [] Scalar
cooling = enthalpy(inlet) - enthalpy(outlet) - condensate * liquid_heat * temperature(outlet)
reheating = enthalpy(reheated[-1]) - enthalpy(outlet)
supply_humidity = relative_humidity(reheated[-1])

def passing():
    for index in range(path.shape[0]):
        yield path[index]

show_animation(render.conditioning(relative_humidity, enthalpy, path, stack((inlet, outlet)), directions, passing()),
               "air_conditioning", duration_ms=65)

The air crosses the coil, its state running left at fixed water content to the saturation curve and then down along it while water collects in the tray, and crosses the reheater, running right at fixed water content to the supply temperature. The ideal coil has no air passing it uncooled.

In [ ]:
print(f"cooling: {float(cooling.to_array()):.1f} kJ, water removed: {1000 * float(condensate.to_array()):.1f} g, "
      f"reheating: {float(reheating.to_array()):.1f} kJ, per kg dry air")
print(f"supply relative humidity: {float(supply_humidity.to_array()):.0%}")

In [ ]:
# checks
np.testing.assert_allclose((humidity(inlet) - humidity(path[-1]) - condensate).kernel, 0, atol=1e-12)
np.testing.assert_allclose((enthalpy(inlet) - enthalpy(path[-1]) - cooling - condensate * liquid_heat * coil_temperature
                            + reheating).kernel, 0, atol=1e-10)
assert np.all(relative_humidity(path).to_array() <= 1 + 1e-10)
np.testing.assert_allclose(temperature(directions).to_array(), 1, atol=1e-10)
np.testing.assert_allclose(humidity(directions[0]).to_array(), 0, atol=1e-10)
np.testing.assert_allclose(derivative(relative_humidity)(outlet)(directions[1]).to_array(), 0, atol=1e-10)